# Notebook 02: Model Training, Validation & Evaluation
## Daily Energy Consumption Forecasting Using Machine Learning

In this notebook, we:
1. Engineer calendar, autoregressive lag, and shifted rolling statistics.
2. Perform chronological train, validation, and holdout test splitting.
3. Train Naive baseline, Seasonal Naive, Linear Regression, and Random Forest models.
4. Evaluate out-of-sample performance using MAE, RMSE, and MAPE.
5. Execute 7, 14, and 30-day recursive forecasts.

In [ ]:
import sys
import os
sys.path.append('../')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from src.features import engineer_all_features, FEATURE_COLUMNS, TARGET_COLUMN
from src.train import split_chronologically, train_and_evaluate_all
from src.forecast import generate_recursive_forecast
import joblib

### 1. Load Data & Engineer Leakage-Free Features

In [ ]:
df_raw = pd.read_csv('../data/processed/daily_energy.csv')
feat_df = engineer_all_features(df_raw, drop_incomplete=True)

print(f"Feature matrix shape: {feat_df.shape}")
print("Engineered features:", FEATURE_COLUMNS)
feat_df.head(3)

### 2. Execute Training & Chronological Evaluation

In [ ]:
artifacts, comp_table = train_and_evaluate_all(
    '../data/processed/daily_energy.csv',
    models_dir='../models',
    outputs_dir='../outputs'
)

print("\n--- Out-of-Sample Test Evaluation Results ---")
comp_table

### 3. Visualize Actual vs. Predicted on Test Period

In [ ]:
test_preds = pd.read_csv('../outputs/test_predictions.csv')
test_preds['date'] = pd.to_datetime(test_preds['date'])

plt.figure(figsize=(14, 5))
plt.plot(test_preds['date'], test_preds['daily_energy_kwh'], label='Actual Consumption', color='#0f172a', linewidth=1.5)
plt.plot(test_preds['date'], test_preds['rf_pred'], label='Random Forest (MAE 3.82)', color='#10b981', linewidth=1.5, linestyle='--')
plt.plot(test_preds['date'], test_preds['lr_pred'], label='Linear Regression (MAE 4.11)', color='#0284c7', linewidth=1.2, linestyle=':')
plt.title('Out-of-Sample Holdout Forecast vs. Ground Truth (Jul - Nov 2010)', fontsize=13, fontweight='bold')
plt.xlabel('Date')
plt.ylabel('Daily Energy (kWh)')
plt.legend()
plt.show()

### 4. Recursive Multi-Step Forecast (7, 14, 30 Days Ahead)

In [ ]:
rf_model = joblib.load('../models/random_forest.joblib')
forecast_30d = generate_recursive_forecast(rf_model, feat_df, horizon_days=30)

forecast_30d['date'] = pd.to_datetime(forecast_30d['date'])
plt.figure(figsize=(12, 4.5))
plt.plot(forecast_30d['date'], forecast_30d['forecast_energy_kwh'], marker='o', color='#0284c7', label='Predicted Daily Energy (kWh)')
plt.fill_between(forecast_30d['date'], forecast_30d['lower_bound_kwh'], forecast_30d['upper_bound_kwh'], color='#0284c7', alpha=0.2, label='95% Prediction Interval')
plt.title('30-Day Recursive Multi-Step Forecast with Expanding Uncertainty', fontsize=13, fontweight='bold')
plt.xlabel('Forecast Date')
plt.ylabel('Energy Consumption (kWh)')
plt.legend()
plt.show()